# Examples and proofs of the work shown in the "Vacuum FEM Discretization" section of my dissertation work

$(L + B) \vec{u} = \frac{1}{k} C \vec{u} \\$
$L_{a, a'} = \int_{\Omega} D(\vec{x}) (\nabla \Lambda_{a}(\vec{x}) \cdot \nabla \Lambda_{a'}(\vec{x})) dV + \int_{d \Omega} \frac{1}{2} \Lambda_{a}(\vec{x}) \Lambda_{a'}(\vec{x}) ds \\$
$B_{a, a'} = \int_\Omega \Sigma_a(\vec{x})\Lambda_{a}(\vec{x}) \Lambda_{a'}(\vec{x}) \; d \Omega \\$
$C_{a, a'} = \int_\Omega \nu\Sigma_f(\vec{x})\Lambda_{a}(\vec{x}) \Lambda_{a'}(\vec{x}) \; d \Omega$

We separate the diffusion matrix, $L$, into the volume integral and the surface integral term

$L = S_D + R_v$

with

$S_D = \sum_{i} S_{ii}$ where $i$ is a dimension ($x$, $y$, or $z$)

$S_{ij} = \begin{bmatrix}
    \int_{\Omega} D(\vec{x}) \frac{\partial \Lambda_0(\vec{x})}{\partial i} \frac{\partial \Lambda_0(\vec{x})}{\partial j} dV & \int_{\Omega} D(\vec{x}) \frac{\partial \Lambda_0(\vec{x})}{\partial i} \frac{\partial \Lambda_1(\vec{x})}{\partial j} dV & \ldots \\
    \int_{\Omega} D(\vec{x}) \frac{\partial \Lambda_1(\vec{x})}{\partial i} \frac{\partial \Lambda_0(\vec{x})}{\partial j} dV & \int_{\Omega} D(\vec{x}) \frac{\partial \Lambda_1(\vec{x})}{\partial i} \frac{\partial \Lambda_1(\vx)}{\partial j} dV & \ldots \\
    \vdots & \vdots & \ddots
    \end{bmatrix}$

$ R_v = \frac{1}{2}\begin{bmatrix}
        \int_{d\Omega} \Lambda_0(\vec{x}) \Lambda_0(\vec{x}) ds & \int_{d\Omega} \Lambda_0(\vec{x}) \Lambda_1(\vec{x}) ds & \ldots \\
        \int_{d\Omega} \Lambda_1(\vec{x}) \Lambda_0(\vec{x}) ds & \int_{d\Omega} \Lambda_1(\vec{x}) \Lambda_1(\vec{x}) ds & \ldots \\
        \vdots & \vdots & \ddots
    \end{bmatrix} $

# Goals of this Notebook:
- Show that we can efficiently block-encode the BPX preconditioned stiffness matrix $F^T S_D F$
- Show that we can efficiently block-encode the BPX preconditioned surface integral matrix $F^T R_v F$
- Show that the condition numbers/singular values of these matrices (and their sum) allows us to apply the pseudoinverse of their sum efficiently

In [1]:
import sys
import numpy as np
import os
sys.path.append(os.getcwd())
import scipy as sp
import scipy.sparse as spsp
from scipy.sparse import csr_matrix, coo_matrix
import itertools
from fast_inversion.BPX import FEM_BPX_helpers as FEM